# KBA Course Advisor - RAG + Deterministic Rule Engine

**What this notebook demonstrates**

| Stage | Module (`src/course_advisor/`) |
|---|---|
| Knowledge base: parse -> chunk | `knowledge_base.py` |
| Embeddings + retrieval | `embeddings.py`, `retriever.py` |
| Rule checking (Python, no LLM) | `rules.py` |
| Recommendation engine | `recommender.py` |
| Chatbot (pretrained LLM optional) | `chatbot.py`, `llm.py` |
| Report rendering | `report.py` |

> **Data notice:** no official college documents were available, so the knowledge base in `kb/` is **fictional sample data**; every file is labelled as such.

**Design principle:** the LLM never decides eligibility. Python rules decide; retrieval supplies the evidence shown to the student; the LLM (when configured) only phrases the answer from those facts.

In [1]:
import sys, json, textwrap
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import Markdown, display
from course_advisor import config
from course_advisor.knowledge_base import load_documents, chunk_documents, build_courses
from course_advisor.recommender import Advisor
from course_advisor.profiles import load_all_profiles
from course_advisor.report import plan_markdown
from course_advisor.models import Verdict
print("project root:", ROOT)

project root: /home/user/kba-course-advisor


## 1. Knowledge base: document parsing and chunking

In [2]:
docs = load_documents()
chunks = chunk_documents(docs)
print(f"{len(docs)} documents ({sum(d.doc_type=='course' for d in docs)} courses, "
      f"{sum(d.doc_type=='policy' for d in docs)} policies) -> {len(chunks)} chunks")
print("\n--- raw file: kb/courses/CS304.md ---")
print((config.COURSES_DIR / "CS304.md").read_text())

34 documents (30 courses, 4 policies) -> 135 chunks

--- raw file: kb/courses/CS304.md ---
# CS304 - Machine Learning

> FICTIONAL SAMPLE DATA - invented for the KBA assignment. Not an official college document.

code: CS304
title: Machine Learning
credits: 3
level: 3
offered: Spring
prerequisites: CS201, MATH201, MATH202
corequisites: none
min_credits_passed: 0
min_gpa: none
category: elective
tags: machine learning, data science, python, classification, regression, neural networks
source: fictional_sample

## Overview
CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.

## Description
Introduces supervised and unsupervised learning: linear and logistic regression, decision trees, clustering, neural networks, model evaluation and overfitting, using Python libraries.

## Topics
Regression; classification; clustering; model evaluation; neural network 

In [3]:
print("--- parsed metadata (drives the rule engine) ---")
doc = next(d for d in docs if d.doc_id == "CS304")
print(json.dumps(doc.metadata, indent=2))
print("\n--- chunks produced from this document ---")
for c in chunks:
    if c.doc_id == "CS304":
        print(f"{c.chunk_id:10} [{c.section}] {c.text[:110]}...")
print("\n--- a policy file is chunked by section too ---")
for c in chunks:
    if c.doc_id == "POL-CREDIT":
        print(f"{c.chunk_id:12} [{c.section}] {c.text[:100]}...")

--- parsed metadata (drives the rule engine) ---
{
  "code": "CS304",
  "title": "Machine Learning",
  "credits": "3",
  "level": "3",
  "offered": "Spring",
  "prerequisites": "CS201, MATH201, MATH202",
  "corequisites": "none",
  "min_credits_passed": "0",
  "min_gpa": "none",
  "category": "elective",
  "tags": "machine learning, data science, python, classification, regression, neural networks",
  "source": "fictional_sample"
}

--- chunks produced from this document ---
CS304#0.0  [Overview] CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring sem...
CS304#1.0  [Description] Introduces supervised and unsupervised learning: linear and logistic regression, decision trees, clustering, n...
CS304#2.0  [Topics] Regression; classification; clustering; model evaluation; neural network basics; feature engineering....
CS304#3.0  [Registration notes] Standard registration rules apply; see the registration policies....

--- a policy file 

## 2. Embeddings and retrieval
Default embedder: TF-IDF (unigrams+bigrams, cosine similarity) - runs offline. A pretrained `sentence-transformers` model can be swapped in with `Advisor.from_knowledge_base(make_embedder('sbert'))`.

In [4]:
advisor = Advisor.from_knowledge_base()
print("embedder:", advisor.retriever.embedder.name, "| matrix:", advisor.retriever.matrix.shape)
for q in ["machine learning and neural networks", "maximum credit hours for a student on probation"]:
    print(f"\nQUERY: {q}")
    for h in advisor.retriever.retrieve(q, 3):
        print(f"  {h.score:.2f}  {h.chunk.source_file} § {h.chunk.section}")

embedder: tfidf | matrix: (135, 1828)

QUERY: machine learning and neural networks
  0.49  kb/courses/CS402.md § Registration notes
  0.48  kb/courses/CS304.md § Registration notes
  0.44  kb/courses/CS402.md § Description

QUERY: maximum credit hours for a student on probation
  0.48  kb/policies/credit_load_policy.md § Maximum credit hours per semester
  0.24  kb/policies/credit_load_policy.md § Exceeding the limit
  0.24  kb/policies/credit_load_policy.md § Minimum load


## 3. Rule engine (deterministic Python)
The rules read structured fields parsed from the same documents the retriever indexes.

In [5]:
from course_advisor.models import StudentProfile
demo = StudentProfile("Demo", level=2, term="Spring", passed_courses=["CS101","CS102","MATH101","CS103"], gpa=2.9)
for code_ in ["CS201", "CS304", "CS206", "CS101", "CS303L"]:
    v = advisor.rules.violations(demo, code_)
    print(f"{code_:7}", "ELIGIBLE" if not v else "; ".join(f"{x.rule.value}: {x.message}" for x in v))
print("\nmax credits:", advisor.rules.max_credits(demo))

CS201   ELIGIBLE
CS304   LEVEL_RESTRICTION: CS304 is a level 3 course; student is level 2; PREREQUISITE_UNMET: missing prerequisite(s): CS201, MATH201, MATH202
CS206   NOT_OFFERED_THIS_TERM: CS206 is offered only in Fall, not in Spring; PREREQUISITE_UNMET: missing prerequisite(s): CS201
CS101   ALREADY_PASSED: CS101 has already been passed
CS303L  LEVEL_RESTRICTION: CS303L is a level 3 course; student is level 2; COREQUISITE_UNAVAILABLE: corequisite CS303 cannot be registered (CS303 is a level 3 course; student is level 2; missing prerequisite(s): CS202)

max credits: (18, 'GPA 2.90 (standard limit)')


## 4. Test profiles
Seven synthetic students (`profiles/*.json`): levels 1-4, one with **unmet prerequisites** (Youssef), one that **exceeds the credit limit** (Sara), and edge cases (Karim). For each: input, retrieved evidence, accepted courses, rejected courses with reasons, final recommendation.

In [6]:
profiles = load_all_profiles()
plans = [advisor.plan(p) for p in profiles]
import pandas as pd
display(pd.DataFrame([{"student": p.name, "level": p.level, "GPA": p.gpa, "preferred": p.preferred_credits,
                       "limit": pl.max_credits, "total": pl.total_credits, "recommended": ", ".join(pl.selected),
                       "rejected": sum(d.verdict is Verdict.REJECTED for d in pl.decisions.values())}
                      for p, pl in zip(profiles, plans)]))

,student,level,GPA,preferred,limit,total,recommended,rejected
0,Amira Hassan,1,NaN,15,18,13,"CS101, MATH101, ENG101, HUM101",26
1,Omar Khaled,1,3.2,15,21,12,"CS102, ENG101, MATH102, HUM101",23
2,Layla Mansour,2,3.4,18,21,13,"MATH202, CS203, CS202, CS204",16
3,Youssef Adel,3,3.0,18,21,16,"CS301, MATH202, CS202, CS307, CS204",14
4,Sara Nabil,3,2.4,24,18,16,"CS307, CS303, CS301, CS302, CS305, CS303L",11
5,Nour Fathy,4,3.7,21,21,19,"CS405, CS403, CS407, CS401, CS306, CS304",4
6,Karim Samir,4,1.8,18,12,12,"CS203, MATH201, MATH202, HUM101",15


### Profile 1

In [7]:
display(Markdown(plan_markdown(plans[0], advisor.courses)))

# Test Profile: Amira Hassan

**Scenario:** New level-1 student starting in Spring with no history: availability (Fall-only CS103) and prerequisite chain rules apply.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 1 |
| Upcoming term | Spring |
| GPA | none (first semester) |
| Passed courses | none |
| Skills | basic computer use |
| Interests | programming, problem solving |
| Goals | become a software developer |
| Preferred credit load | 15 |
| Requested courses | - |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS101 (RECOMMENDED)**
> **[kb/courses/CS101.md § Overview]** (sim 0.59) CS101 Introduction to Programming is a 4-credit-hour core course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.
> **[kb/courses/CS101.md § Registration notes]** (sim 0.42) Standard registration rules apply; see the registration policies.

**MATH101 (RECOMMENDED)**
> **[kb/courses/MATH101.md § Overview]** (sim 0.55) MATH101 Calculus I is a 3-credit-hour core course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.
> **[kb/courses/MATH101.md § Description]** (sim 0.05) Covers limits, continuity, derivatives and basic integration of single-variable functions, with applications to optimisation and rates of change.

**ENG101 (RECOMMENDED)**
> **[kb/courses/ENG101.md § Overview]** (sim 0.72) ENG101 Technical Writing and Communication is a 3-credit-hour core course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.

**HUM101 (RECOMMENDED)**
> **[kb/courses/HUM101.md § Overview]** (sim 0.69) HUM101 Critical Thinking and Society is a 3-credit-hour general education course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.

**CS102 (REJECTED)**
> **[kb/courses/CS102.md § Overview]** (sim 0.68) CS102 Object-Oriented Programming is a 3-credit-hour core course at academic level 1. It is offered in the Spring semester only. Registration requirements - prerequisites: CS101.

**CS103 (REJECTED)**
> **[kb/courses/CS103.md § Overview]** (sim 0.66) CS103 Digital Logic Design is a 3-credit-hour core course at academic level 1. It is offered in the Fall semester only. There are no prerequisites or other registration requirements.

**MATH102 (REJECTED)**
> **[kb/courses/MATH102.md § Overview]** (sim 0.62) MATH102 Discrete Mathematics is a 3-credit-hour core course at academic level 1. It is offered in the Spring semester only. Registration requirements - prerequisites: MATH101.

**CS202 (REJECTED)**
> **[kb/courses/CS202.md § Overview]** (sim 0.65) CS202 Computer Organization and Architecture is a 4-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS103.

**CS303L (REJECTED)**
> **[kb/courses/CS303L.md § Overview]** (sim 0.58) CS303L Computer Networks Laboratory is a 1-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - corequisite: CS303.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 18 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/level_and_availability_policy.md § Academic level restriction]** (sim 0.47) Courses are numbered by academic level from 1 to 4. A student may register only for courses at or below the academic level recorded for the student by the registrar. A level 1 student may not register for level 2, 3 or 4 courses, and a level 2 student may not register for level 3 or 4 courses.

> **[kb/policies/prerequisite_policy.md § Corequisites]** (sim 0.38) A corequisite must be passed before, or registered in the same semester as, the course that requires it. If the corequisite course is not passed, the student must register for both courses together, and both must be individually eligible.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS101 | Introduction to Programming | 4 | No prerequisites required. Offered in Spring; level 1 is within student level 1. Core degree requirement. Matches your interests/goals: programming, problem solving. |
| MATH101 | Calculus I | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 1. Core degree requirement. |
| ENG101 | Technical Writing and Communication | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 1. Core degree requirement. |
| HUM101 | Critical Thinking and Society | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 1. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS102 | Object-Oriented Programming | PREREQUISITE_UNMET | missing prerequisite(s): CS101. **Hint:** Take CS101 first (eligible now) to work towards CS102. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS103 | Digital Logic Design | NOT_OFFERED_THIS_TERM | CS103 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| MATH102 | Discrete Mathematics | PREREQUISITE_UNMET | missing prerequisite(s): MATH101. **Hint:** Take MATH101 first (eligible now) to work towards MATH102. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS202 | Computer Organization and Architecture | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS202 is a level 2 course; student is level 1; missing prerequisite(s): CS103. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS303L | Computer Networks Laboratory | LEVEL_RESTRICTION, COREQUISITE_UNAVAILABLE | CS303L is a level 3 course; student is level 1; corequisite CS303 cannot be registered (CS303 is a level 3 course; student is level 1; missing prerequisite(s): CS202). | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Corequisites |
| CS301 | Design and Analysis of Algorithms | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS301 is a level 3 course; student is level 1; missing prerequisite(s): CS201, MATH102. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS204 | Web Application Development | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS204 is a level 2 course; student is level 1; missing prerequisite(s): CS102. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS302 | Operating Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS302 is a level 3 course; student is level 1; missing prerequisite(s): CS201, CS202. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS303 | Computer Networks | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS303 is a level 3 course; student is level 1; missing prerequisite(s): CS202. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS201 | Data Structures | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS201 is a level 2 course; student is level 1; missing prerequisite(s): CS102. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |

_(16 further courses are also ineligible; see the JSON output for the full list.)_

## 5. Final recommendation

**Register for:** CS101, MATH101, ENG101, HUM101  
**Total:** 13 credit hours (maximum allowed: 18, planned: 15)



### Profile 2

In [8]:
display(Markdown(plan_markdown(plans[1], advisor.courses)))

# Test Profile: Omar Khaled

**Scenario:** Level-1 student who asks for a level-2 web course: level restriction must reject it and the remaining level-1 courses fill the plan.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 1 |
| Upcoming term | Spring |
| GPA | 3.2 |
| Passed courses | CS101, MATH101, CS103 |
| Skills | python |
| Interests | object oriented programming, web development |
| Goals | build web applications |
| Preferred credit load | 15 |
| Requested courses | CS204 |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS102 (RECOMMENDED)**
> **[kb/courses/CS102.md § Overview]** (sim 0.68) CS102 Object-Oriented Programming is a 3-credit-hour core course at academic level 1. It is offered in the Spring semester only. Registration requirements - prerequisites: CS101.
> **[kb/courses/CS102.md § Registration notes]** (sim 0.49) Standard registration rules apply; see the registration policies.

**ENG101 (RECOMMENDED)**
> **[kb/courses/ENG101.md § Overview]** (sim 0.72) ENG101 Technical Writing and Communication is a 3-credit-hour core course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.

**MATH102 (RECOMMENDED)**
> **[kb/courses/MATH102.md § Overview]** (sim 0.62) MATH102 Discrete Mathematics is a 3-credit-hour core course at academic level 1. It is offered in the Spring semester only. Registration requirements - prerequisites: MATH101.

**HUM101 (RECOMMENDED)**
> **[kb/courses/HUM101.md § Overview]** (sim 0.69) HUM101 Critical Thinking and Society is a 3-credit-hour general education course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.

**CS204 (REJECTED)**
> **[kb/courses/CS204.md § Overview]** (sim 0.63) CS204 Web Application Development is a 3-credit-hour elective course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS102.

**MATH202 (REJECTED)**
> **[kb/courses/MATH202.md § Overview]** (sim 0.63) MATH202 Linear Algebra is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.

**CS202 (REJECTED)**
> **[kb/courses/CS202.md § Overview]** (sim 0.65) CS202 Computer Organization and Architecture is a 4-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS103.

**MATH201 (REJECTED)**
> **[kb/courses/MATH201.md § Overview]** (sim 0.64) MATH201 Probability and Statistics is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.

**CS306 (REJECTED)**
> **[kb/courses/CS306.md § Overview]** (sim 0.62) CS306 Mobile Application Development is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS204.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 21 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/level_and_availability_policy.md § Academic level restriction]** (sim 0.47) Courses are numbered by academic level from 1 to 4. A student may register only for courses at or below the academic level recorded for the student by the registrar. A level 1 student may not register for level 2, 3 or 4 courses, and a level 2 student may not register for level 3 or 4 courses.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/prerequisite_policy.md § Corequisites]** (sim 0.38) A corequisite must be passed before, or registered in the same semester as, the course that requires it. If the corequisite course is not passed, the student must register for both courses together, and both must be individually eligible.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS102 | Object-Oriented Programming | 3 | Prerequisites satisfied: CS101 passed. Offered in Spring; level 1 is within student level 1. Core degree requirement. Matches your interests/goals: object oriented programming. |
| ENG101 | Technical Writing and Communication | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 1. Core degree requirement. |
| MATH102 | Discrete Mathematics | 3 | Prerequisites satisfied: MATH101 passed. Offered in Spring; level 1 is within student level 1. Core degree requirement. |
| HUM101 | Critical Thinking and Society | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 1. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS204 | Web Application Development | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS204 is a level 2 course; student is level 1; missing prerequisite(s): CS102. **Hint:** Take CS102 first (eligible now) to work towards CS204. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| MATH202 | Linear Algebra | LEVEL_RESTRICTION | MATH202 is a level 2 course; student is level 1. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS202 | Computer Organization and Architecture | LEVEL_RESTRICTION | CS202 is a level 2 course; student is level 1. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| MATH201 | Probability and Statistics | LEVEL_RESTRICTION | MATH201 is a level 2 course; student is level 1. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS306 | Mobile Application Development | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS306 is a level 3 course; student is level 1; missing prerequisite(s): CS204. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS305 | Cybersecurity Fundamentals | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS305 is a level 3 course; student is level 1; missing prerequisite(s): CS202. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS301 | Design and Analysis of Algorithms | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS301 is a level 3 course; student is level 1; missing prerequisite(s): CS201, MATH102. **Hint:** Take MATH102 first (eligible now) to work towards CS301. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS303L | Computer Networks Laboratory | LEVEL_RESTRICTION, COREQUISITE_UNAVAILABLE | CS303L is a level 3 course; student is level 1; corequisite CS303 cannot be registered (CS303 is a level 3 course; student is level 1; missing prerequisite(s): CS202). | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Corequisites |
| CS304 | Machine Learning | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS304 is a level 3 course; student is level 1; missing prerequisite(s): CS201, MATH201, MATH202. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS302 | Operating Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS302 is a level 3 course; student is level 1; missing prerequisite(s): CS201, CS202. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS201 | Data Structures | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS201 is a level 2 course; student is level 1; missing prerequisite(s): CS102. **Hint:** Take CS102 first (eligible now) to work towards CS201. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |

_(12 further courses are also ineligible; see the JSON output for the full list.)_

## 5. Final recommendation

**Register for:** CS102, ENG101, MATH102, HUM101  
**Total:** 12 credit hours (maximum allowed: 21, planned: 15)



### Profile 3

In [9]:
display(Markdown(plan_markdown(plans[2], advisor.courses)))

# Test Profile: Layla Mansour

**Scenario:** Level-2 student with strong ML interest: machine-learning courses are level 3, so interests cannot override the level rule.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 2 |
| Upcoming term | Spring |
| GPA | 3.4 |
| Passed courses | CS101, MATH101, ENG101, HUM101, CS103, CS102, MATH102, CS201, MATH201, CS205 |
| Skills | python, statistics |
| Interests | machine learning, data science |
| Goals | become an AI engineer |
| Preferred credit load | 18 |
| Requested courses | - |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**MATH202 (RECOMMENDED)**
> **[kb/courses/MATH202.md § Overview]** (sim 0.63) MATH202 Linear Algebra is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.
> **[kb/courses/MATH202.md § Description]** (sim 0.13) Covers vectors, matrices, linear systems, vector spaces, eigenvalues and applications to computing and data science.

**CS203 (RECOMMENDED)**
> **[kb/courses/CS203.md § Overview]** (sim 0.56) CS203 Database Systems is a 3-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS102.
> **[kb/courses/CS203.md § Registration notes]** (sim 0.04) Standard registration rules apply; see the registration policies.

**CS202 (RECOMMENDED)**
> **[kb/courses/CS202.md § Overview]** (sim 0.65) CS202 Computer Organization and Architecture is a 4-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS103.

**CS204 (RECOMMENDED)**
> **[kb/courses/CS204.md § Overview]** (sim 0.63) CS204 Web Application Development is a 3-credit-hour elective course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS102.

**CS407 (REJECTED)**
> **[kb/courses/CS407.md § Overview]** (sim 0.70) CS407 Professional Ethics and Law in Computing is a 3-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - at least 30 passed credit hours.

**CS206 (REJECTED)**
> **[kb/courses/CS206.md § Overview]** (sim 0.61) CS206 Computer Graphics is a 3-credit-hour elective course at academic level 2. It is offered in the Fall semester only. Registration requirements - prerequisites: CS201.

**CS301 (REJECTED)**
> **[kb/courses/CS301.md § Overview]** (sim 0.59) CS301 Design and Analysis of Algorithms is a 3-credit-hour core course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH102.

**CS307 (REJECTED)**
> **[kb/courses/CS307.md § Overview]** (sim 0.59) CS307 Artificial Intelligence is a 3-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH201.

**CS304 (REJECTED)**
> **[kb/courses/CS304.md § Overview]** (sim 0.54) CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 21 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/level_and_availability_policy.md § Academic level restriction]** (sim 0.47) Courses are numbered by academic level from 1 to 4. A student may register only for courses at or below the academic level recorded for the student by the registrar. A level 1 student may not register for level 2, 3 or 4 courses, and a level 2 student may not register for level 3 or 4 courses.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/prerequisite_policy.md § Corequisites]** (sim 0.38) A corequisite must be passed before, or registered in the same semester as, the course that requires it. If the corequisite course is not passed, the student must register for both courses together, and both must be individually eligible.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| MATH202 | Linear Algebra | 3 | Prerequisites satisfied: MATH101 passed. Offered in Spring; level 2 is within student level 2. Core degree requirement. |
| CS203 | Database Systems | 3 | Prerequisites satisfied: CS102 passed. Offered in Spring; level 2 is within student level 2. Core degree requirement. Matches your interests/goals: data science. |
| CS202 | Computer Organization and Architecture | 4 | Prerequisites satisfied: CS103 passed. Offered in Spring; level 2 is within student level 2. Core degree requirement. |
| CS204 | Web Application Development | 3 | Prerequisites satisfied: CS102 passed. Offered in Spring; level 2 is within student level 2. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS407 | Professional Ethics and Law in Computing | LEVEL_RESTRICTION | CS407 is a level 4 course; student is level 2. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS206 | Computer Graphics | NOT_OFFERED_THIS_TERM | CS206 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS301 | Design and Analysis of Algorithms | LEVEL_RESTRICTION | CS301 is a level 3 course; student is level 2. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS307 | Artificial Intelligence | LEVEL_RESTRICTION | CS307 is a level 3 course; student is level 2. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS304 | Machine Learning | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS304 is a level 3 course; student is level 2; missing prerequisite(s): MATH202. **Hint:** Take MATH202 first (eligible now) to work towards CS304. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS402 | Deep Learning | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS402 is a level 4 course; student is level 2; missing prerequisite(s): CS304. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS403 | Distributed Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS403 is a level 4 course; student is level 2; missing prerequisite(s): CS302, CS303. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS302 | Operating Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS302 is a level 3 course; student is level 2; missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS302. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS303 | Computer Networks | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS303 is a level 3 course; student is level 2; missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS303. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS303L | Computer Networks Laboratory | LEVEL_RESTRICTION, COREQUISITE_UNAVAILABLE | CS303L is a level 3 course; student is level 2; corequisite CS303 cannot be registered (CS303 is a level 3 course; student is level 2; missing prerequisite(s): CS202). | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Corequisites |

_(6 further courses are also ineligible; see the JSON output for the full list.)_

## 5. Final recommendation

**Register for:** MATH202, CS203, CS202, CS204  
**Total:** 13 credit hours (maximum allowed: 21, planned: 18)



### Profile 4

In [10]:
display(Markdown(plan_markdown(plans[3], advisor.courses)))

# Test Profile: Youssef Adel

**Scenario:** UNMET PREREQUISITES: transfer student who skipped CS202 and MATH202 but asks for Operating Systems and Machine Learning.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 3 |
| Upcoming term | Spring |
| GPA | 3.0 |
| Passed courses | CS101, MATH101, ENG101, HUM101, CS103, CS102, MATH102, CS201, MATH201, CS205, CS203 |
| Skills | python, sql |
| Interests | operating systems, machine learning |
| Goals | work on systems software |
| Preferred credit load | 18 |
| Requested courses | CS302, CS304 |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS301 (RECOMMENDED)**
> **[kb/courses/CS301.md § Overview]** (sim 0.59) CS301 Design and Analysis of Algorithms is a 3-credit-hour core course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH102.

**MATH202 (RECOMMENDED)**
> **[kb/courses/MATH202.md § Overview]** (sim 0.63) MATH202 Linear Algebra is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.
> **[kb/courses/MATH202.md § Description]** (sim 0.04) Covers vectors, matrices, linear systems, vector spaces, eigenvalues and applications to computing and data science.

**CS202 (RECOMMENDED)**
> **[kb/courses/CS202.md § Overview]** (sim 0.65) CS202 Computer Organization and Architecture is a 4-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS103.

**CS307 (RECOMMENDED)**
> **[kb/courses/CS307.md § Overview]** (sim 0.59) CS307 Artificial Intelligence is a 3-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH201.

**CS204 (RECOMMENDED)**
> **[kb/courses/CS204.md § Overview]** (sim 0.63) CS204 Web Application Development is a 3-credit-hour elective course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS102.

**CS302 (REJECTED)**
> **[kb/courses/CS302.md § Overview]** (sim 0.59) CS302 Operating Systems is a 3-credit-hour core course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, CS202.

**CS304 (REJECTED)**
> **[kb/courses/CS304.md § Overview]** (sim 0.54) CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.

**CS206 (REJECTED)**
> **[kb/courses/CS206.md § Overview]** (sim 0.61) CS206 Computer Graphics is a 3-credit-hour elective course at academic level 2. It is offered in the Fall semester only. Registration requirements - prerequisites: CS201.

**CS303 (REJECTED)**
> **[kb/courses/CS303.md § Overview]** (sim 0.52) CS303 Computer Networks is a 3-credit-hour core course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS202.

**CS303L (REJECTED)**
> **[kb/courses/CS303L.md § Overview]** (sim 0.58) CS303L Computer Networks Laboratory is a 1-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - corequisite: CS303.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 21 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/prerequisite_policy.md § Corequisites]** (sim 0.38) A corequisite must be passed before, or registered in the same semester as, the course that requires it. If the corequisite course is not passed, the student must register for both courses together, and both must be individually eligible.

> **[kb/policies/level_and_availability_policy.md § Academic level restriction]** (sim 0.47) Courses are numbered by academic level from 1 to 4. A student may register only for courses at or below the academic level recorded for the student by the registrar. A level 1 student may not register for level 2, 3 or 4 courses, and a level 2 student may not register for level 3 or 4 courses.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS301 | Design and Analysis of Algorithms | 3 | Prerequisites satisfied: CS201, MATH102 passed. Offered in Spring; level 3 is within student level 3. Core degree requirement. |
| MATH202 | Linear Algebra | 3 | Prerequisites satisfied: MATH101 passed. Offered in Spring; level 2 is within student level 3. Core degree requirement. |
| CS202 | Computer Organization and Architecture | 4 | Prerequisites satisfied: CS103 passed. Offered in Spring; level 2 is within student level 3. Core degree requirement. |
| CS307 | Artificial Intelligence | 3 | Prerequisites satisfied: CS201, MATH201 passed. Offered in Spring; level 3 is within student level 3. |
| CS204 | Web Application Development | 3 | Prerequisites satisfied: CS102 passed. Offered in Spring; level 2 is within student level 3. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS302 | Operating Systems | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS302. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS304 | Machine Learning | PREREQUISITE_UNMET | missing prerequisite(s): MATH202. **Hint:** Take MATH202 first (eligible now) to work towards CS304. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS206 | Computer Graphics | NOT_OFFERED_THIS_TERM | CS206 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS303 | Computer Networks | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS303. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS303L | Computer Networks Laboratory | COREQUISITE_UNAVAILABLE | corequisite CS303 cannot be registered (missing prerequisite(s): CS202). | kb/policies/prerequisite_policy.md § Corequisites |
| CS305 | Cybersecurity Fundamentals | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS305. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS306 | Mobile Application Development | PREREQUISITE_UNMET | missing prerequisite(s): CS204. **Hint:** Take CS204 first (eligible now) to work towards CS306. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS407 | Professional Ethics and Law in Computing | LEVEL_RESTRICTION | CS407 is a level 4 course; student is level 3. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS402 | Deep Learning | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS402 is a level 4 course; student is level 3; missing prerequisite(s): CS304. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS403 | Distributed Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS403 is a level 4 course; student is level 3; missing prerequisite(s): CS302, CS303. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS405 | Advanced Cybersecurity | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS405 is a level 4 course; student is level 3; missing prerequisite(s): CS305. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS406 | Natural Language Processing | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS406 is a level 4 course; student is level 3; missing prerequisite(s): CS304. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |

_(2 further courses are also ineligible; see the JSON output for the full list.)_

## 5. Final recommendation

**Register for:** CS301, MATH202, CS202, CS307, CS204  
**Total:** 16 credit hours (maximum allowed: 21, planned: 18)



### Profile 5

In [11]:
display(Markdown(plan_markdown(plans[4], advisor.courses)))

# Test Profile: Sara Nabil

**Scenario:** EXCEEDS CREDIT LIMIT: GPA 2.4 allows 18 credit hours but the student wants 24 and requests 22 hours of courses.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 3 |
| Upcoming term | Spring |
| GPA | 2.4 |
| Passed courses | CS101, MATH101, ENG101, HUM101, CS103, CS102, MATH102, CS201, MATH201, MATH202, CS205, CS202, CS203 |
| Skills | python, java |
| Interests | networking, security, artificial intelligence |
| Goals | graduate on time with broad skills |
| Preferred credit load | 24 |
| Requested courses | CS301, CS302, CS303, CS303L, CS304, CS305, CS307, CS204 |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS307 (RECOMMENDED)**
> **[kb/courses/CS307.md § Overview]** (sim 0.59) CS307 Artificial Intelligence is a 3-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH201.
> **[kb/courses/CS307.md § Registration notes]** (sim 0.38) Standard registration rules apply; see the registration policies.

**CS303 (RECOMMENDED)**
> **[kb/courses/CS303.md § Overview]** (sim 0.52) CS303 Computer Networks is a 3-credit-hour core course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS202.
> **[kb/courses/CS303.md § Registration notes]** (sim 0.07) Standard registration rules apply; see the registration policies.

**CS301 (RECOMMENDED)**
> **[kb/courses/CS301.md § Overview]** (sim 0.59) CS301 Design and Analysis of Algorithms is a 3-credit-hour core course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS201, MATH102.

**CS302 (RECOMMENDED)**
> **[kb/courses/CS302.md § Overview]** (sim 0.59) CS302 Operating Systems is a 3-credit-hour core course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, CS202.

**CS305 (RECOMMENDED)**
> **[kb/courses/CS305.md § Overview]** (sim 0.58) CS305 Cybersecurity Fundamentals is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS202.
> **[kb/courses/CS305.md § Registration notes]** (sim 0.10) Standard registration rules apply; see the registration policies.

**CS303L (RECOMMENDED)**
> **[kb/courses/CS303L.md § Overview]** (sim 0.58) CS303L Computer Networks Laboratory is a 1-credit-hour elective course at academic level 3. It is offered in every semester (Fall and Spring). Registration requirements - corequisite: CS303.
> **[kb/courses/CS303L.md § Registration notes]** (sim 0.07) Standard registration rules apply; see the registration policies.

**CS204 (REJECTED)**
> **[kb/courses/CS204.md § Overview]** (sim 0.63) CS204 Web Application Development is a 3-credit-hour elective course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS102.

**CS304 (REJECTED)**
> **[kb/courses/CS304.md § Overview]** (sim 0.54) CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.

**CS206 (REJECTED)**
> **[kb/courses/CS206.md § Overview]** (sim 0.61) CS206 Computer Graphics is a 3-credit-hour elective course at academic level 2. It is offered in the Fall semester only. Registration requirements - prerequisites: CS201.

**CS306 (REJECTED)**
> **[kb/courses/CS306.md § Overview]** (sim 0.62) CS306 Mobile Application Development is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS204.

**CS407 (REJECTED)**
> **[kb/courses/CS407.md § Overview]** (sim 0.70) CS407 Professional Ethics and Law in Computing is a 3-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - at least 30 passed credit hours.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 18 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/level_and_availability_policy.md § Academic level restriction]** (sim 0.47) Courses are numbered by academic level from 1 to 4. A student may register only for courses at or below the academic level recorded for the student by the registrar. A level 1 student may not register for level 2, 3 or 4 courses, and a level 2 student may not register for level 3 or 4 courses.

> **[kb/policies/level_and_availability_policy.md § Minimum passed credit hours]** (sim 0.52) Senior courses may require a minimum number of passed credit hours, stated in the course overview. The Capstone Project requires 60 passed credit hours. Professional Ethics and Law in Computing requires 30 passed credit hours.

> **[kb/policies/prerequisite_policy.md § Additional requirements]** (sim 0.39) Some courses require a minimum number of passed credit hours or a minimum cumulative GPA. These requirements are stated in the course overview and must be satisfied at the time of registration.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS307 | Artificial Intelligence | 3 | Requested by the student and passes every registration rule. Prerequisites satisfied: CS201, MATH201 passed. Offered in Spring; level 3 is within student level 3. Matches your interests/goals: artificial intelligence. |
| CS303 | Computer Networks | 3 | Corequisite of CS303L; must be registered together. Requested by the student and passes every registration rule. Prerequisites satisfied: CS202 passed. Offered in Spring; level 3 is within student level 3. Core degree requirement. Matches your interests/goals: networking. |
| CS301 | Design and Analysis of Algorithms | 3 | Requested by the student and passes every registration rule. Prerequisites satisfied: CS201, MATH102 passed. Offered in Spring; level 3 is within student level 3. Core degree requirement. |
| CS302 | Operating Systems | 3 | Requested by the student and passes every registration rule. Prerequisites satisfied: CS201, CS202 passed. Offered in Spring; level 3 is within student level 3. Core degree requirement. |
| CS305 | Cybersecurity Fundamentals | 3 | Requested by the student and passes every registration rule. Prerequisites satisfied: CS202 passed. Offered in Spring; level 3 is within student level 3. Matches your interests/goals: security. |
| CS303L | Computer Networks Laboratory | 1 | Requested by the student and passes every registration rule. No prerequisites required. Offered in Spring; level 3 is within student level 3. Matches your interests/goals: networking. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS204 | Web Application Development | CREDIT_LIMIT_EXCEEDED | adding CS204 (3 credit hours) would bring the load to 19, above the maximum of 18. | kb/policies/credit_load_policy.md § Exceeding the limit |
| CS304 | Machine Learning | CREDIT_LIMIT_EXCEEDED | adding CS304 (3 credit hours) would bring the load to 19, above the maximum of 18. | kb/policies/credit_load_policy.md § Exceeding the limit |
| CS206 | Computer Graphics | NOT_OFFERED_THIS_TERM | CS206 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS306 | Mobile Application Development | PREREQUISITE_UNMET | missing prerequisite(s): CS204. **Hint:** Take CS204 first (eligible now) to work towards CS306. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS407 | Professional Ethics and Law in Computing | LEVEL_RESTRICTION | CS407 is a level 4 course; student is level 3. | kb/policies/level_and_availability_policy.md § Academic level restriction |
| CS405 | Advanced Cybersecurity | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS405 is a level 4 course; student is level 3; missing prerequisite(s): CS305. **Hint:** Take CS305 first (eligible now) to work towards CS405. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS403 | Distributed Systems | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS403 is a level 4 course; student is level 3; missing prerequisite(s): CS302, CS303. **Hint:** Take CS302, CS303 first (eligible now) to work towards CS403. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS406 | Natural Language Processing | LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS406 is a level 4 course; student is level 3; missing prerequisite(s): CS304. **Hint:** Take CS304 first (eligible now) to work towards CS406. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Prerequisites |
| CS401 | Capstone Project I | LEVEL_RESTRICTION, PREREQUISITE_UNMET, MIN_PASSED_CREDITS | CS401 is a level 4 course; student is level 3; missing prerequisite(s): CS301; requires 60 passed credit hours; student has 42. **Hint:** Take CS301 first (eligible now) to work towards CS401. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/level_and_availability_policy.md § Minimum passed credit hours; kb/policies/prerequisite_policy.md § Prerequisites |
| CS402 | Deep Learning | LEVEL_RESTRICTION, PREREQUISITE_UNMET, MIN_GPA | CS402 is a level 4 course; student is level 3; missing prerequisite(s): CS304; requires GPA >= 2.5; student GPA is 2.40. **Hint:** Take CS304 first (eligible now) to work towards CS402. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/prerequisite_policy.md § Additional requirements; kb/policies/prerequisite_policy.md § Prerequisites |
| CS404 | Cloud Computing | NOT_OFFERED_THIS_TERM, LEVEL_RESTRICTION, PREREQUISITE_UNMET | CS404 is offered only in Fall, not in Spring; CS404 is a level 4 course; student is level 3; missing prerequisite(s): CS303. **Hint:** Take CS303 first (eligible now) to work towards CS404. | kb/policies/level_and_availability_policy.md § Academic level restriction; kb/policies/level_and_availability_policy.md § Semester availability; kb/policies/prerequisite_policy.md § Prerequisites |

## 5. Final recommendation

**Register for:** CS307, CS303, CS301, CS302, CS305, CS303L  
**Total:** 16 credit hours (maximum allowed: 18, planned: 18)

- WARNING: Preferred load of 24 credit hours exceeds the maximum of 18 allowed for this student (GPA 2.40 (standard limit)); the plan is capped at 18.


### Profile 6

In [12]:
display(Markdown(plan_markdown(plans[5], advisor.courses)))

# Test Profile: Nour Fathy

**Scenario:** High-GPA senior eligible for the capstone: requests a Fall-only cloud course (rejected) and an advanced security course.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 4 |
| Upcoming term | Spring |
| GPA | 3.7 |
| Passed courses | CS101, MATH101, ENG101, HUM101, CS103, CS102, MATH102, CS201, MATH201, MATH202, CS205, CS204, CS202, CS203, CS301, CS303, CS303L, CS302, CS305, CS307 |
| Skills | python, networking, cryptography |
| Interests | cybersecurity, distributed systems, cloud computing |
| Goals | security engineer in a cloud company |
| Preferred credit load | 21 |
| Requested courses | CS404, CS405 |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS405 (RECOMMENDED)**
> **[kb/courses/CS405.md § Overview]** (sim 0.57) CS405 Advanced Cybersecurity is a 3-credit-hour elective course at academic level 4. It is offered in the Spring semester only. Registration requirements - prerequisites: CS305.
> **[kb/courses/CS405.md § Registration notes]** (sim 0.11) Standard registration rules apply; see the registration policies.

**CS403 (RECOMMENDED)**
> **[kb/courses/CS403.md § Overview]** (sim 0.58) CS403 Distributed Systems is a 3-credit-hour elective course at academic level 4. It is offered in the Spring semester only. Registration requirements - prerequisites: CS302, CS303.
> **[kb/courses/CS403.md § Registration notes]** (sim 0.42) Standard registration rules apply; see the registration policies.

**CS407 (RECOMMENDED)**
> **[kb/courses/CS407.md § Overview]** (sim 0.70) CS407 Professional Ethics and Law in Computing is a 3-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - at least 30 passed credit hours.

**CS401 (RECOMMENDED)**
> **[kb/courses/CS401.md § Overview]** (sim 0.61) CS401 Capstone Project I is a 4-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS205, CS301; at least 60 passed credit hours; cumulative GPA of at least 2.0.

**CS306 (RECOMMENDED)**
> **[kb/courses/CS306.md § Overview]** (sim 0.62) CS306 Mobile Application Development is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS204.
> **[kb/courses/CS306.md § Description]** (sim 0.04) Designs and builds cross-platform mobile applications with attention to user interface, device sensors and cloud back ends.

**CS304 (RECOMMENDED)**
> **[kb/courses/CS304.md § Overview]** (sim 0.54) CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.
> **[kb/courses/CS304.md § Description]** (sim 0.03) Introduces supervised and unsupervised learning: linear and logistic regression, decision trees, clustering, neural networks, model evaluation and overfitting, using Python libraries.

**CS404 (REJECTED)**
> **[kb/courses/CS404.md § Overview]** (sim 0.60) CS404 Cloud Computing is a 3-credit-hour elective course at academic level 4. It is offered in the Fall semester only. Registration requirements - prerequisites: CS303.

**CS206 (REJECTED)**
> **[kb/courses/CS206.md § Overview]** (sim 0.61) CS206 Computer Graphics is a 3-credit-hour elective course at academic level 2. It is offered in the Fall semester only. Registration requirements - prerequisites: CS201.

**CS402 (REJECTED)**
> **[kb/courses/CS402.md § Overview]** (sim 0.58) CS402 Deep Learning is a 3-credit-hour elective course at academic level 4. It is offered in the Spring semester only. Registration requirements - prerequisites: CS304; cumulative GPA of at least 2.5.

**CS406 (REJECTED)**
> **[kb/courses/CS406.md § Overview]** (sim 0.67) CS406 Natural Language Processing is a 3-credit-hour elective course at academic level 4. It is offered in the Spring semester only. Registration requirements - prerequisites: CS304.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 21 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS405 | Advanced Cybersecurity | 3 | Requested by the student and passes every registration rule. Prerequisites satisfied: CS305 passed. Offered in Spring; level 4 is within student level 4. Matches your interests/goals: cybersecurity. |
| CS403 | Distributed Systems | 3 | Prerequisites satisfied: CS302, CS303 passed. Offered in Spring; level 4 is within student level 4. Matches your interests/goals: distributed systems, cloud computing. |
| CS407 | Professional Ethics and Law in Computing | 3 | No prerequisites required. Offered in Spring; level 4 is within student level 4. Core degree requirement. Matches your interests/goals: cloud computing. |
| CS401 | Capstone Project I | 4 | Prerequisites satisfied: CS205, CS301 passed. Offered in Spring; level 4 is within student level 4. Core degree requirement. |
| CS306 | Mobile Application Development | 3 | Prerequisites satisfied: CS204 passed. Offered in Spring; level 3 is within student level 4. |
| CS304 | Machine Learning | 3 | Prerequisites satisfied: CS201, MATH201, MATH202 passed. Offered in Spring; level 3 is within student level 4. Matches your interests/goals: python. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS404 | Cloud Computing | NOT_OFFERED_THIS_TERM | CS404 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS206 | Computer Graphics | NOT_OFFERED_THIS_TERM | CS206 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS402 | Deep Learning | PREREQUISITE_UNMET | missing prerequisite(s): CS304. **Hint:** Take CS304 first (eligible now) to work towards CS402. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS406 | Natural Language Processing | PREREQUISITE_UNMET | missing prerequisite(s): CS304. **Hint:** Take CS304 first (eligible now) to work towards CS406. | kb/policies/prerequisite_policy.md § Prerequisites |

## 5. Final recommendation

**Register for:** CS405, CS403, CS407, CS401, CS306, CS304  
**Total:** 19 credit hours (maximum allowed: 21, planned: 21)



### Profile 7

In [13]:
display(Markdown(plan_markdown(plans[6], advisor.courses)))

# Test Profile: Karim Samir

**Scenario:** EDGE CASES: probation GPA (limit 12), only 29 passed credits (capstone needs 60, ethics needs 30), GPA below capstone minimum, and a request to repeat a passed course.

## 1. Student input

| Field | Value |
|---|---|
| Academic level | 4 |
| Upcoming term | Spring |
| GPA | 1.8 |
| Passed courses | CS101, MATH101, ENG101, CS103, CS102, MATH102, CS201, CS205, CS301 |
| Skills | java |
| Interests | software engineering, project management |
| Goals | finish the degree and start the capstone |
| Preferred credit load | 18 |
| Requested courses | CS401, CS407, CS101 |

## 2. Retrieved evidence from the Knowledge Base

### 2a. Course passages (retrieved per course)

**CS203 (RECOMMENDED)**
> **[kb/courses/CS203.md § Overview]** (sim 0.56) CS203 Database Systems is a 3-credit-hour core course at academic level 2. It is offered in the Spring semester only. Registration requirements - prerequisites: CS102.

**MATH201 (RECOMMENDED)**
> **[kb/courses/MATH201.md § Overview]** (sim 0.64) MATH201 Probability and Statistics is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.

**MATH202 (RECOMMENDED)**
> **[kb/courses/MATH202.md § Overview]** (sim 0.63) MATH202 Linear Algebra is a 3-credit-hour core course at academic level 2. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: MATH101.

**HUM101 (RECOMMENDED)**
> **[kb/courses/HUM101.md § Overview]** (sim 0.69) HUM101 Critical Thinking and Society is a 3-credit-hour general education course at academic level 1. It is offered in every semester (Fall and Spring). There are no prerequisites or other registration requirements.

**CS407 (REJECTED)**
> **[kb/courses/CS407.md § Overview]** (sim 0.70) CS407 Professional Ethics and Law in Computing is a 3-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - at least 30 passed credit hours.

**CS401 (REJECTED)**
> **[kb/courses/CS401.md § Overview]** (sim 0.61) CS401 Capstone Project I is a 4-credit-hour core course at academic level 4. It is offered in every semester (Fall and Spring). Registration requirements - prerequisites: CS205, CS301; at least 60 passed credit hours; cumulative GPA of at least 2.0.

**CS302 (REJECTED)**
> **[kb/courses/CS302.md § Overview]** (sim 0.59) CS302 Operating Systems is a 3-credit-hour core course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, CS202.

**CS405 (REJECTED)**
> **[kb/courses/CS405.md § Overview]** (sim 0.57) CS405 Advanced Cybersecurity is a 3-credit-hour elective course at academic level 4. It is offered in the Spring semester only. Registration requirements - prerequisites: CS305.

**CS304 (REJECTED)**
> **[kb/courses/CS304.md § Overview]** (sim 0.54) CS304 Machine Learning is a 3-credit-hour elective course at academic level 3. It is offered in the Spring semester only. Registration requirements - prerequisites: CS201, MATH201, MATH202.

### 2b. Policy passages (retrieved for the rules applied)

Credit limit for this student: 12 credit hours.

> **[kb/policies/credit_load_policy.md § Exceeding the limit]** (sim 0.38) A registration that would exceed the maximum credit hours is rejected. Exceptions are not granted by the advising system; students must petition the dean in writing.

> **[kb/policies/level_and_availability_policy.md § Minimum passed credit hours]** (sim 0.52) Senior courses may require a minimum number of passed credit hours, stated in the course overview. The Capstone Project requires 60 passed credit hours. Professional Ethics and Law in Computing requires 30 passed credit hours.

> **[kb/policies/prerequisite_policy.md § Additional requirements]** (sim 0.39) Some courses require a minimum number of passed credit hours or a minimum cumulative GPA. These requirements are stated in the course overview and must be satisfied at the time of registration.

> **[kb/policies/prerequisite_policy.md § Prerequisites]** (sim 0.52) A student may register for a course only if every prerequisite listed for that course has been passed. Prerequisites are cumulative: all listed prerequisites are required, not just one. A course that is currently in progress or was failed does not satisfy a prerequisite.

> **[kb/policies/level_and_availability_policy.md § Semester availability]** (sim 0.58) Each course is offered in the Fall semester, the Spring semester, or both. A course can only be registered in a semester in which it is offered. Course overviews state the semesters in which the course is offered. A course offered only in Fall cannot be registered in the Spring semester.

> **[kb/policies/prerequisite_policy.md § Corequisites]** (sim 0.38) A corequisite must be passed before, or registered in the same semester as, the course that requires it. If the corequisite course is not passed, the student must register for both courses together, and both must be individually eligible.

## 3. Accepted courses

| Course | Title | Credits | Reasons |
|---|---|---|---|
| CS203 | Database Systems | 3 | Prerequisites satisfied: CS102 passed. Offered in Spring; level 2 is within student level 4. Core degree requirement. |
| MATH201 | Probability and Statistics | 3 | Prerequisites satisfied: MATH101 passed. Offered in Spring; level 2 is within student level 4. Core degree requirement. |
| MATH202 | Linear Algebra | 3 | Prerequisites satisfied: MATH101 passed. Offered in Spring; level 2 is within student level 4. Core degree requirement. |
| HUM101 | Critical Thinking and Society | 3 | No prerequisites required. Offered in Spring; level 1 is within student level 4. |

## 4. Rejected courses and reasons

| Course | Title | Rule failed | Reason | Policy source |
|---|---|---|---|---|
| CS407 | Professional Ethics and Law in Computing | MIN_PASSED_CREDITS | requires 30 passed credit hours; student has 29. | kb/policies/level_and_availability_policy.md § Minimum passed credit hours |
| CS401 | Capstone Project I | MIN_PASSED_CREDITS, MIN_GPA | requires 60 passed credit hours; student has 29; requires GPA >= 2.0; student GPA is 1.80. | kb/policies/level_and_availability_policy.md § Minimum passed credit hours; kb/policies/prerequisite_policy.md § Additional requirements |
| CS302 | Operating Systems | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS302. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS405 | Advanced Cybersecurity | PREREQUISITE_UNMET | missing prerequisite(s): CS305. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS304 | Machine Learning | PREREQUISITE_UNMET | missing prerequisite(s): MATH201, MATH202. **Hint:** Take MATH201, MATH202 first (eligible now) to work towards CS304. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS306 | Mobile Application Development | PREREQUISITE_UNMET | missing prerequisite(s): CS204. **Hint:** Take CS204 first (eligible now) to work towards CS306. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS206 | Computer Graphics | NOT_OFFERED_THIS_TERM | CS206 is offered only in Fall, not in Spring. | kb/policies/level_and_availability_policy.md § Semester availability |
| CS303 | Computer Networks | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS303. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS303L | Computer Networks Laboratory | COREQUISITE_UNAVAILABLE | corequisite CS303 cannot be registered (missing prerequisite(s): CS202). | kb/policies/prerequisite_policy.md § Corequisites |
| CS305 | Cybersecurity Fundamentals | PREREQUISITE_UNMET | missing prerequisite(s): CS202. **Hint:** Take CS202 first (eligible now) to work towards CS305. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS307 | Artificial Intelligence | PREREQUISITE_UNMET | missing prerequisite(s): MATH201. **Hint:** Take MATH201 first (eligible now) to work towards CS307. | kb/policies/prerequisite_policy.md § Prerequisites |
| CS403 | Distributed Systems | PREREQUISITE_UNMET | missing prerequisite(s): CS302, CS303. | kb/policies/prerequisite_policy.md § Prerequisites |

_(3 further courses are also ineligible; see the JSON output for the full list.)_

Eligible but not selected (would exceed the planned load):
CS202 (4 cr), CS204 (3 cr)

## 5. Final recommendation

**Register for:** CS203, MATH201, MATH202, HUM101  
**Total:** 12 credit hours (maximum allowed: 12, planned: 12)

- WARNING: Preferred load of 18 credit hours exceeds the maximum of 12 allowed for this student (GPA 1.80 < 2.0 (probation limit)); the plan is capped at 12.


## 5. Chatbot
The chatbot parses preference changes deterministically, re-runs the rule engine, and (if `ANTHROPIC_API_KEY` is set) lets a pretrained LLM phrase the reply from the grounded facts. If the LLM mentions a course that is not in the facts, the reply is discarded and the grounded template is used. Below it runs in offline template mode.

In [14]:
from course_advisor.chatbot import AdvisorChat
from course_advisor.llm import default_llm
llm = default_llm()
print("LLM:", llm.name if llm else "none (offline grounded-template mode)")
chat = AdvisorChat(advisor, profiles[3], llm)      # Youssef: unmet prerequisites
for msg in ["Why can't I take CS302?", "I want 12 credits", "I am interested in cybersecurity",
            "show me alternative plans", "What is the maximum number of credits on probation?"]:
    r = chat.handle(msg)
    print(f"\nSTUDENT: {msg}\nADVISOR: {r.text}")
    if r.sources: print("SOURCES:", "; ".join(r.sources))

LLM: none (offline grounded-template mode)

STUDENT: Why can't I take CS302?
ADVISOR: CS302 (Operating Systems) is NOT allowed: missing prerequisite(s): CS202 [kb/courses/CS302.md § Overview, kb/policies/prerequisite_policy.md § Prerequisites]
SOURCES: kb/courses/CS302.md § Overview; kb/policies/prerequisite_policy.md § Prerequisites

STUDENT: I want 12 credits
ADVISOR: Preferences updated: preferred load set to 12 credit hours.
Recommended plan for Youssef Adel (12/21 credit hours):
- CS301 Design and Analysis of Algorithms (3 cr): Prerequisites satisfied: CS201, MATH102 passed. Offered in Spring; level 3 is within student level 3.
- MATH202 Linear Algebra (3 cr): Prerequisites satisfied: MATH101 passed. Offered in Spring; level 2 is within student level 3.
- CS307 Artificial Intelligence (3 cr): Prerequisites satisfied: CS201, MATH201 passed. Offered in Spring; level 3 is within student level 3.
- CS204 Web Application Development (3 cr): Prerequisites satisfied: CS102 passed. Offere

## 6. Verification
The automated suite (`pytest`, 80+ tests) checks the invariants independently of the planner, including a seeded fuzz over random student histories: no recommended course ever violates a rule, the credit limit is never exceeded, and every recommendation has retrieved evidence.

In [15]:
import subprocess
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-600:])

........................................................................ [ 87%]
..........                                                               [100%]
82 passed in 4.23s

